# Telco customer churn: effect size and risk profile

## Goal
Measure how strongly each variable is associated with churn, and describe a risk profile that states its sample size.
1. **Effect size**: a chi-square test says whether an association exists; Cramér's V says how strong it is.
2. **Risk profile**: combine features that stood out in the exploratory analysis and see how far the churn rate of that customer group is above the overall rate, with the number of customers.

## Ground rules
- **Training set only**, as in [07-odds-ratios](07-odds-ratios.ipynb). The test set is not used and the frozen threshold and recipe are not changed.
- Notebook 02 ran a chi-square test on `Contract` with all 7,043 customers; here it is redone on the training set, so the numbers differ slightly.
- Everything is a **descriptive association**. Profiles built on small groups are unstable, so the group size is always reported.

## Setup
Load the raw CSV, reproduce the same 80/20 split as notebooks 06 and 07, and drop the test set. No model is fitted.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split

# Project folder = the nearest parent that contains reports/final-plan.json
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reports" / "final-plan.json").exists())

df = pd.read_csv(PROJECT / "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
assert df["customerID"].is_unique

y = df["Churn"].copy()
x = df.drop(columns=["Churn", "customerID"]).copy()
x["TotalCharges"] = pd.to_numeric(x["TotalCharges"], errors="coerce")

# Same split as notebooks 06 and 07; only the training set is kept
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=2002, stratify=y
)
assert len(x_train) == 5634
del x_test, y_test

# Features and target back in one table, for cross-tabulation and filtering
train = x_train.copy()
train["Churn"] = y_train
overall_rate = (train["Churn"] == "Yes").mean()
print("Train:", len(train), "| overall churn rate", round(overall_rate, 4))

Train: 5634 | overall churn rate 0.2654


## 1. Effect size: Cramér's V
1. Cross-tabulate the variable against `Churn`.
2. Run the chi-square test on that table.
3. Compute

```text
V = sqrt( chi-square / (n × (min(rows, columns) − 1)) )
```

where `n` is the sum of all cells. V lies between 0 and 1; a common rough guide is about 0.1 weak, 0.3 moderate, 0.5 strong.

First for `Contract`, then for `InternetService`, `PaymentMethod` and tenure (grouped into bands, since it is numeric). `gender` is included as a control with almost no association.

In [2]:
table = pd.crosstab(train["Contract"], train["Churn"])
result = chi2_contingency(table)
print(f"Chi-square statistic: {result.statistic:.2f}")

n = table.values.sum()          # total customers
k = min(table.shape) - 1        # table.shape is (rows, columns)

v = np.sqrt(result.statistic / (n * k))

print("V =", round(v, 3))

Chi-square statistic: 946.84
V = 0.41


In [3]:
train["tenure_group"] = pd.cut(train["tenure"], bins=[-1, 12, 24, 48, 72],
                               labels=["0-12", "13-24", "25-48", "49-72"])

def cramers_v(col):
    table = pd.crosstab(train[col], train["Churn"])
    result = chi2_contingency(table)
    n = table.values.sum()
    k = min(table.shape) - 1
    v = np.sqrt(result.statistic / (n * k))
    return v

for col in ["Contract", "tenure_group", "InternetService", "PaymentMethod", "gender"]:
    print(col, round(cramers_v(col), 3))

Contract 0.41
tenure_group 0.353
InternetService 0.324
PaymentMethod 0.303
gender 0.016


## 2. Risk profile
A customer group defined by features with high churn in the exploratory analysis: how many customers are in it, what is its churn rate, and how does that compare with the overall 26.5%?

The profile combines two conditions, `Contract == "Month-to-month"` and `tenure <= 12`. Adding more conditions usually raises the churn rate further but shrinks the group and makes the figure less stable. Customers on one-year or two-year contracts serve as the low-risk comparison.

In [4]:
mask = (train["Contract"] == "Month-to-month") & (train["tenure"] <= 12)
print("High-risk profile: customers", mask.sum())
print("High-risk profile: churn rate", round((train.loc[mask, "Churn"] == "Yes").mean(), 3))

contrast = (train["Contract"] != "Month-to-month")
print("One-year or two-year contract: customers", contrast.sum())
print("One-year or two-year contract: churn rate", round((train.loc[contrast, "Churn"] == "Yes").mean(), 3))

High-risk profile: customers 1597
High-risk profile: churn rate 0.518
One-year or two-year contract: customers 2520
One-year or two-year contract: churn rate 0.067


## Checks
- V uses the total of the cross-table and `min(rows, columns) − 1`.
- Every profile is reported with its number of customers, not only its churn rate.
- Only the training set is used.
- The conclusions are stated as associations.

## Findings
On the training set, contract type has the strongest association with churn (V 0.41), followed by tenure band (0.35), internet service (0.32) and payment method (0.30). Gender is almost unrelated (0.02).

Month-to-month customers with a tenure of 12 months or less number 1,597 and have a churn rate of 51.8%; customers on one-year or two-year contracts number 2,520 and have a churn rate of 6.7%. This is a descriptive comparison of two customer groups. It does not show that moving a customer to a longer contract would reduce churn.